# Phase 2 - Baseline implementation (Option A)

Paper: Appiah et al. (2023), Extremely randomised trees machine learning model for electricity theft detection.
Data: SGCC dataset from https://github.com/henryRDlab/ElectricityTheftDetection (extract data.z01, data.z02, data.zip and put data.csv next to this notebook).

- E1: authors' pipeline as in the paper (preprocessing + SMOTETomek on all data before the split)
- E2: same thing but split first, so the test set keeps the real theft rate
- E3: my corrected pipeline from Phase 1 (ET, RF, LR and ET with summary features only)

SEED = 42 for everything.

In [1]:
import time, json, os, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.metrics import (matthews_corrcoef, average_precision_score, roc_auc_score, precision_score,
                             recall_score, f1_score, accuracy_score, confusion_matrix, precision_recall_curve, make_scorer)
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
warnings.filterwarnings("ignore")
# settings
SEED = 42
N_JOBS = -1   # use every CPU core (results do not depend on the number of cores)
OUT = "p2out"; os.makedirs(OUT, exist_ok=True)
RESULTS = {}
T0 = time.time()
# print progress with elapsed time, also saved to progress.txt
def log(msg):
    line = f"[{time.time()-T0:7.0f}s] {msg}"
    print(line, flush=True)
    with open("progress.txt", "a") as fh: fh.write(line + "\n")
import sklearn, imblearn; print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__, "| numpy", np.__version__)

scikit-learn 1.9.1 | imbalanced-learn 0.14.2 | numpy 2.5.3


## 1. Load data
Sort the day columns by date, add the missing day (18 Sep 2016) and remove empty customers and duplicates. This gives the same counts as the paper (36,677 normal, 3,579 theft).

In [2]:
# load data
df = pd.read_csv("data.csv")
y_all = df["FLAG"].to_numpy()
X_raw = df.drop(columns=["CONS_NO", "FLAG"])
X_raw.columns = pd.to_datetime(X_raw.columns, format="%Y/%m/%d")
full_days = pd.date_range("2014-01-01", "2016-10-31")
X_raw = X_raw.reindex(columns=full_days)          # sorts columns and inserts 18 Sep 2016 as NaN
A = X_raw.to_numpy(dtype=float)
# empty customers (all NaN or 0) and duplicate rows
empty = np.all(np.isnan(A) | (A == 0), axis=1)
dup = pd.DataFrame(A).fillna(-1).duplicated().to_numpy()
print("customers:", len(A), "| theft:", y_all.sum(), "| days:", A.shape[1])
print("empty customers:", empty.sum(), "| duplicate rows among non-empty:", (dup & ~empty).sum())
# drop them
keep = ~empty & ~dup
A = A[keep]; y = y_all[keep]
print("after reduction:", len(A), "customers | theft:", y.sum(), f"({y.mean():.2%})")
lost_n = int(((~keep) & (y_all == 0)).sum()); lost_t = int(((~keep) & (y_all == 1)).sum())
print(f"removed: {lost_n:,} normal ({lost_n/(y_all==0).sum():.1%}) and {lost_t} theft ({lost_t/(y_all==1).sum():.1%})")
# Phase 1 suggested flagging empty meters with a rule instead of deleting them. How good would that rule be?
alert_prec = y_all[empty].mean(); alert_rec = y_all[empty].sum() / y_all.sum()
print(f"empty-meter alert: {empty.sum():,} customers flagged, {int(y_all[empty].sum())} thieves -> precision {alert_prec:.1%} (random choice: {y_all.mean():.1%}), recall {alert_rec:.1%}")
RESULTS["data"] = dict(n_all=int(len(y_all)), n=int(len(y)), theft=int(y.sum()), rate=float(y.mean()), days=int(A.shape[1]),
                       lost_normal=lost_n, lost_theft=lost_t, empty=int(empty.sum()), alert_precision=float(alert_prec), alert_recall=float(alert_rec))

customers: 42372 | theft: 3615 | days: 1035
empty customers: 2114 | duplicate rows among non-empty: 2
after reduction: 40256 customers | theft: 3579 (8.89%)
removed: 2,080 normal (5.4%) and 36 theft (1.0%)
empty-meter alert: 2,114 customers flagged, 36 thieves -> precision 1.7% (random choice: 8.5%), recall 1.0%


## 2. Metrics
Same metrics as the paper, plus PR-AUC (better when theft is rare) and precision in the top 5% (if only the top 5% most suspicious customers get inspected).

In [3]:
# all metrics for one set of predictions
def evaluate(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    k = max(1, int(round(0.05 * len(y_true))))
    top = np.argsort(-prob, kind="stable")[:k]
    return dict(MCC=matthews_corrcoef(y_true, pred), PR_AUC=average_precision_score(y_true, prob),
                ROC_AUC=roc_auc_score(y_true, prob), Precision=precision_score(y_true, pred, zero_division=0),
                Recall=recall_score(y_true, pred), F1=f1_score(y_true, pred), FPR=fp / (fp + tn),
                Accuracy=accuracy_score(y_true, pred), Prec_top5=float(y_true[top].mean()),
                TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), threshold=float(thr), theft_rate=float(y_true.mean()), n=int(len(y_true)))

# one line print of the main metrics
def show(name, m):
    print(f"{name:42s} MCC {m['MCC']:.4f} | PR-AUC {m['PR_AUC']:.4f} | ROC-AUC {m['ROC_AUC']:.4f} | P {m['Precision']:.3f} R {m['Recall']:.3f} FPR {m['FPR']:.3f} | top5% {m['Prec_top5']:.3f}")

# threshold with the best MCC
def best_mcc_threshold(y_true, prob):
    grid = np.unique(np.quantile(prob, np.linspace(0.5, 0.999, 300)))
    scores = [matthews_corrcoef(y_true, (prob >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(scores))])

### 2.1 Phase 1 calculation check
The paper's test set is 50/50, so FPR = recall * (1 - precision) / precision. Applying that to the real class sizes gives the precision we would actually get.

In [4]:
fpr_paper = 0.98 * (1 - 0.97) / 0.97
# 3615 theft / 38757 normal in the full data
tp_ = 0.98 * 3615; fp_ = fpr_paper * 38757
print(f"implied FPR {fpr_paper:.4f} | true alarms {tp_:,.0f} | false alarms {fp_:,.0f} | precision at the real ratio {tp_/(tp_+fp_):.3f}")
RESULTS["phase1_check"] = dict(fpr=fpr_paper, tp=tp_, fp=fp_, precision=tp_/(tp_+fp_))

implied FPR 0.0303 | true alarms 3,543 | false alarms 1,175 | precision at the real ratio 0.751


## 3. Authors' preprocessing
- Eq. 1: missing day = average of the day before and after, otherwise 0
- cap values above mean + 2 std (per customer, the paper doesn't say so I followed Zheng et al. 2018)
- min-max scaling per day

In [5]:
# Eq. 1 - mean of the neighbour days, otherwise 0
def eq1_fill(X):
    X = X.copy(); nan = np.isnan(X)
    prev = np.c_[np.full((len(X), 1), np.nan), X[:, :-1]]
    nxt = np.c_[X[:, 1:], np.full((len(X), 1), np.nan)]
    both = nan & ~np.isnan(prev) & ~np.isnan(nxt)
    X[both] = (prev[both] + nxt[both]) / 2
    X[np.isnan(X)] = 0.0
    return X
# cap at mean + 2 std per customer
def cap_2sigma(X):
    mu = X.mean(1, keepdims=True); sd = X.std(1, keepdims=True)
    return np.minimum(X, mu + 2 * sd)
log("filling and capping (authors' method)")
Xa = cap_2sigma(eq1_fill(A))
print("share of cells that are zero after Equation 1:", round(float((eq1_fill(A) == 0).mean()), 4))

[      2s] filling and capping (authors' method)


share of cells that are zero after Equation 1: 0.3548


## 4. E1 - authors' pipeline (leaky)
Scaling and SMOTETomek on all the data. I used the paper's best settings (500 trees, depth 70) instead of rerunning their grid search.
Checked with an 80/20 split of the resampled data (like the paper) and with 4-fold CV without shuffling (what GridSearchCV(cv=4) does).

In [6]:
log("E1: Min-Max on all data")
# scaling on all data (leaky, same as the paper)
Xs = MinMaxScaler().fit_transform(Xa).astype(np.float32)
log("E1: SMOTETomek on all data (this takes a few minutes)")
Xr, yr = SMOTETomek(random_state=SEED).fit_resample(Xs, y)
print("after SMOTETomek:", len(Xr), "rows | theft:", int(yr.sum()), "| normal:", int((yr == 0).sum()))
RESULTS["E1_resampled"] = dict(rows=int(len(Xr)), theft=int(yr.sum()), normal=int((yr == 0).sum()))

[      3s] E1: Min-Max on all data


[      3s] E1: SMOTETomek on all data (this takes a few minutes)


after SMOTETomek: 73336 rows | theft: 36668 | normal: 36668


In [7]:
# imblearn keeps original rows first (minus any Tomek removals) and appends synthetic rows at the end.
n_real_kept = len(Xr) - (int(yr.sum()) - int(y.sum()))
syn = np.zeros(len(Xr), bool); syn[n_real_kept:] = True
print("synthetic rows:", syn.sum(), f"({syn.mean():.1%} of resampled data)")
# 80/20 split on the resampled data
idx = np.arange(len(Xr))
tr1, te1 = train_test_split(idx, test_size=0.2, stratify=yr, random_state=SEED)
best_params = dict(n_estimators=500, max_depth=70)      # values selected by the paper's grid search
log("E1: Extra Trees with the paper's settings")
gs = ExtraTreesClassifier(random_state=SEED, n_jobs=N_JOBS, **best_params).fit(Xr[tr1], yr[tr1])
p1 = gs.predict_proba(Xr[te1])[:, 1]
m_e1 = evaluate(yr[te1], p1)
# same model, only real customers from the test part
real_te = te1[~syn[te1]]
m_e1_real = evaluate(yr[real_te], gs.predict_proba(Xr[real_te])[:, 1])
show("E1 resampled 50/50 test (paper-style)", m_e1)
show("E1 same model, real customers in test only", m_e1_real)
RESULTS["E1"] = dict(params=best_params, test=m_e1, test_real_only=m_e1_real,
                     test_synthetic_theft_share=float(syn[te1][yr[te1] == 1].mean()))

synthetic rows: 33089 (45.1% of resampled data)
[     23s] E1: Extra Trees with the paper's settings


E1 resampled 50/50 test (paper-style)      MCC 0.9312 | PR-AUC 0.9931 | ROC-AUC 0.9915 | P 0.969 R 0.962 FPR 0.031 | top5% 1.000
E1 same model, real customers in test only MCC 0.8344 | PR-AUC 0.9667 | ROC-AUC 0.9925 | P 0.739 R 0.976 FPR 0.031 | top5% 0.993


In [8]:
# 4-fold CV on the resampled data in file order, as GridSearchCV(cv=4) would do. 100 trees to save time.
log("E1: 4-fold CV without shuffling")
fold_info = []
for k, (a, b) in enumerate(StratifiedKFold(4).split(Xr, yr)):
    m = ExtraTreesClassifier(n_estimators=100, max_depth=70, random_state=SEED, n_jobs=N_JOBS).fit(Xr[a], yr[a])
    pb = m.predict_proba(Xr[b])[:, 1]
    th = b[yr[b] == 1]
    fold_info.append(dict(fold=k + 1, MCC=float(matthews_corrcoef(yr[b], (pb >= 0.5).astype(int))),
                          real_thieves=int((~syn[th]).sum()), synthetic_thieves=int(syn[th].sum())))
    print(fold_info[-1])
RESULTS["E1"]["cv4_file_order"] = fold_info
del Xr, yr

[     44s] E1: 4-fold CV without shuffling


{'fold': 1, 'MCC': 0.9214595946182883, 'real_thieves': 3579, 'synthetic_thieves': 5588}


{'fold': 2, 'MCC': 0.9274639434304371, 'real_thieves': 0, 'synthetic_thieves': 9167}


{'fold': 3, 'MCC': 0.917769436721299, 'real_thieves': 0, 'synthetic_thieves': 9167}


{'fold': 4, 'MCC': 0.9311729265022596, 'real_thieves': 0, 'synthetic_thieves': 9167}


## 5. E2 - split first
Split 80/20 on the real customers first, then scaling + SMOTETomek only on the train part. Test set is not resampled so it has the real theft rate (~8.9%). Same model as E1.

In [9]:
# split first, then scale + resample only the train part
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)
np.save(f"{OUT}/split_test_idx.npy", te)
print("train:", len(tr), "theft", y[tr].sum(), "| test:", len(te), "theft", y[te].sum(), f"({y[te].mean():.2%})")
log("E2: scaling and SMOTETomek on training data only")
sc = MinMaxScaler().fit(Xa[tr])
Xtr2 = sc.transform(Xa[tr]).astype(np.float32); Xte2 = sc.transform(Xa[te]).astype(np.float32)
Xtr2r, ytr2r = SMOTETomek(random_state=SEED).fit_resample(Xtr2, y[tr])
et2 = ExtraTreesClassifier(random_state=SEED, n_jobs=N_JOBS, **best_params).fit(Xtr2r, ytr2r)
p2 = et2.predict_proba(Xte2)[:, 1]
m_e2 = evaluate(y[te], p2)
show("E2 split first, real-ratio test", m_e2)
RESULTS["E2"] = dict(test=m_e2)
np.save(f"{OUT}/p_E2.npy", p2)
del Xtr2, Xtr2r, Xs

train: 32204 theft 2863 | test: 8052 theft 716 (8.89%)
[     60s] E2: scaling and SMOTETomek on training data only


E2 split first, real-ratio test            MCC 0.2951 | PR-AUC 0.3411 | ROC-AUC 0.7793 | P 0.494 R 0.229 FPR 0.023 | top5% 0.442


## 6. E3 - corrected pipeline
Cleaning rules (nothing is fitted here, so it's ok before the split):
1. readings over 10,000 kWh in a day -> missing (meter error)
2. gaps up to 3 days -> linear interpolation, longer gaps stay NaN (ET/RF can handle NaN)
3. log(1 + x), then clip per customer at median + 3*MAD and count the flags
4. summary features: missing share, longest gap, first reading day, missing in 2016, zero days, median/MAD, cv, recent vs overall, flag counts

All of this is per customer. Anything that gets fitted is done on train folds only.

In [10]:
# linear interpolation for gaps of max_gap days or less
def interp_short_gaps(X, max_gap=3):
    X = X.copy()
    for i in range(len(X)):
        r = X[i]; nan = np.isnan(r)
        if not nan.any(): continue
        d = np.diff(np.r_[0, nan.astype(int), 0]); s = np.where(d == 1)[0]; e = np.where(d == -1)[0]
        for a, b in zip(s, e):
            if b - a <= max_gap and a > 0 and b < len(r):
                r[a:b] = np.linspace(r[a - 1], r[b], b - a + 2)[1:-1]
    return X
# longest run of missing days
def longest_run(nan_row):
    d = np.diff(np.r_[0, nan_row.astype(int), 0]); s = np.where(d == 1)[0]; e = np.where(d == -1)[0]
    return int((e - s).max()) if len(s) else 0
log("E3: rule-based cleaning and features")
# >10000 kWh a day -> meter error
C = A.copy(); C[C > 10000] = np.nan
nan0 = np.isnan(C)
C = interp_short_gaps(C, 3)
# log scale, clip high outliers at median + 3*MAD
L = np.log1p(C)
med = np.nanmedian(L, 1, keepdims=True); mad = np.nanmedian(np.abs(L - med), 1, keepdims=True) * 1.4826
hi = med + 3 * mad; lo = med - 3 * mad
n_hi = np.nansum(L > hi, 1); n_lo = np.nansum(L < lo, 1)
L = np.where(L > hi, hi, L)
# summary features per customer
i16 = full_days >= pd.Timestamp("2016-01-01")
with np.errstate(all="ignore"):
    obs = ~np.isnan(C)
    first = np.argmax(obs, 1)
    raw_mean = np.nanmean(C, 1); raw_sd = np.nanstd(C, 1)
    summ = pd.DataFrame({
        "share_missing": nan0.mean(1), "longest_gap": [longest_run(r) for r in nan0], "first_reading_day": first,
        "share_missing_2016": nan0[:, i16].mean(1), "share_zero": np.nansum(C == 0, 1) / obs.sum(1),
        "median_log": med[:, 0], "mad_log": mad[:, 0], "cv": raw_sd / raw_mean,
        "recent_vs_all": np.nanmedian(L[:, -90:], 1) - med[:, 0], "n_high_flags": n_hi, "n_low_flags": n_lo})
summ = summ.replace([np.inf, -np.inf], np.nan)
# daily readings + summary features
X3 = np.hstack([L, summ.to_numpy()]).astype(np.float32)
feat_names = [d.strftime("%Y-%m-%d") for d in full_days] + list(summ.columns)
print("E3 feature matrix:", X3.shape, "| missing cells:", f"{np.isnan(X3).mean():.1%}")
print("readings set to missing as impossible (>10,000 kWh):", int((A > 10000).sum()))

[     98s] E3: rule-based cleaning and features


E3 feature matrix: (40256, 1046) | missing cells: 24.2%
readings set to missing as impossible (>10,000 kWh): 260


### 6.1 Nested CV
- outer: 5 folds, only for estimating performance
- inner: 5 folds for tuning (PR-AUC)
- ET and RF try the same 8 random combos of max_features, min_samples_leaf, max_depth
- LR tries 8 values of C
- 100 trees while searching, 500 for the final models
- class_weight="balanced" instead of SMOTE
- extra ET model with only the 11 summary features
- threshold = best MCC on the out-of-fold train predictions

In [11]:
from sklearn.model_selection import RandomizedSearchCV
Xtr3, Xte3, ytr, yte = X3[tr], X3[te], y[tr], y[te]
np.save(f"{OUT}/y_test.npy", yte)   # lets the figures be redrawn later without rerunning
# outer folds for evaluation, inner folds for tuning
outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
inner = StratifiedKFold(5, shuffle=True, random_state=SEED)
N_ITER = 8
tree_space = {"max_features": ["sqrt", 0.05, 0.1], "min_samples_leaf": [1, 5, 20], "max_depth": [10, 20, None]}
# model, search space, columns to use
MODELS = {
    "ET": (ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=1), tree_space, slice(None)),
    "RF": (RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", random_state=SEED, n_jobs=1), tree_space, slice(None)),
    "LR": (make_pipeline(SimpleImputer(strategy="median"), RobustScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
           {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]}, slice(None)),
    "ET_summary": (ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=1), tree_space, slice(-summ.shape[1], None)),
}

# chosen settings refitted with 500 trees
def final_model(base, params):
    m = clone(base).set_params(**params)
    if "n_estimators" in m.get_params(): m.set_params(n_estimators=500, n_jobs=N_JOBS)   # final fit: trees on all cores
    return m

# random search on the inner folds, returns the best params
def search(base, space, X_, y_):
    return RandomizedSearchCV(clone(base), space, n_iter=N_ITER, scoring="average_precision", cv=inner,
                              random_state=SEED, refit=False, n_jobs=N_JOBS).fit(X_, y_).best_params_   # 8 settings x 5 folds run in parallel
# nested CV -> threshold -> final fit on full train -> test
def run_model(name):
    base, space, cols = MODELS[name]
    Xtr_c, Xte_c = Xtr3[:, cols], Xte3[:, cols]
    oof = np.zeros(len(ytr)); fs = []; chosen = []
    for k, (a, b) in enumerate(outer.split(Xtr_c, ytr)):
        best = search(base, space, Xtr_c[a], ytr[a]); chosen.append(best)
        oof[b] = final_model(base, best).fit(Xtr_c[a], ytr[a]).predict_proba(Xtr_c[b])[:, 1]
        fs.append(dict(PR_AUC=average_precision_score(ytr[b], oof[b]), ROC_AUC=roc_auc_score(ytr[b], oof[b])))
        log(f"{name} outer fold {k+1}: PR-AUC {fs[-1]['PR_AUC']:.4f} | chose {best}")
    thr = best_mcc_threshold(ytr, oof)
    for k, (a, b) in enumerate(outer.split(Xtr_c, ytr)):
        fs[k]["MCC"] = matthews_corrcoef(ytr[b], (oof[b] >= thr).astype(int))
    best_all = search(base, space, Xtr_c, ytr)
    model = final_model(base, best_all).fit(Xtr_c, ytr)
    p = model.predict_proba(Xte_c)[:, 1]
    np.save(f"{OUT}/p_E3_{name}.npy", p)
    out = dict(test=evaluate(yte, p, thr), folds=fs, thr=thr, chosen_per_fold=chosen, final_params=best_all)
    show(f"E3 {name}, test", out["test"]); print("final settings:", best_all)
    return out, model

In [12]:
log("E3: Extra Trees")
res_et, et_final = run_model("ET")
m_et = res_et["test"]; p_et = np.load(f"{OUT}/p_E3_ET.npy")

[    102s] E3: Extra Trees


[    247s] ET outer fold 1: PR-AUC 0.3762 | chose {'min_samples_leaf': 5, 'max_features': 0.1, 'max_depth': 20}


[    379s] ET outer fold 2: PR-AUC 0.3673 | chose {'min_samples_leaf': 5, 'max_features': 0.1, 'max_depth': 20}


[    475s] ET outer fold 3: PR-AUC 0.3802 | chose {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


[    567s] ET outer fold 4: PR-AUC 0.4241 | chose {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


[    663s] ET outer fold 5: PR-AUC 0.3864 | chose {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


E3 ET, test                                MCC 0.3479 | PR-AUC 0.3775 | ROC-AUC 0.7996 | P 0.416 R 0.392 FPR 0.054 | top5% 0.516
final settings: {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


In [13]:
others = {}
for name in ["RF", "LR", "ET_summary"]:
    log(f"E3: {name}")
    others[name], _ = run_model(name)

[    796s] E3: RF


[   1072s] RF outer fold 1: PR-AUC 0.3698 | chose {'min_samples_leaf': 5, 'max_features': 0.1, 'max_depth': 20}


[   1321s] RF outer fold 2: PR-AUC 0.3765 | chose {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


[   1562s] RF outer fold 3: PR-AUC 0.3948 | chose {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


[   1814s] RF outer fold 4: PR-AUC 0.4220 | chose {'min_samples_leaf': 20, 'max_features': 0.1, 'max_depth': 20}


...

[   2819s] ET_summary outer fold 2: PR-AUC 0.3196 | chose {'min_samples_leaf': 1, 'max_features': 0.05, 'max_depth': None}


[   2821s] ET_summary outer fold 3: PR-AUC 0.3592 | chose {'min_samples_leaf': 1, 'max_features': 0.05, 'max_depth': None}


[   2823s] ET_summary outer fold 4: PR-AUC 0.3890 | chose {'min_samples_leaf': 1, 'max_features': 0.05, 'max_depth': None}


[   2826s] ET_summary outer fold 5: PR-AUC 0.3320 | chose {'min_samples_leaf': 1, 'max_features': 0.05, 'max_depth': None}


E3 ET_summary, test                        MCC 0.3173 | PR-AUC 0.3401 | ROC-AUC 0.7815 | P 0.387 R 0.366 FPR 0.057 | top5% 0.442
final settings: {'min_samples_leaf': 1, 'max_features': 0.05, 'max_depth': None}


In [14]:
# mean and std over the outer folds
def cv_summary(fs):
    return {k: [float(np.mean([f[k] for f in fs])), float(np.std([f[k] for f in fs]))] for k in fs[0]}
RESULTS["E3"] = {}
for k, v in [("ET", res_et)] + list(others.items()):
    RESULTS["E3"][k] = dict(test=v["test"], cv=cv_summary(v["folds"]), thr=v["thr"], chosen_per_fold=v["chosen_per_fold"], final_params=v["final_params"])
    print(f"{k:11s} CV PR-AUC {RESULTS['E3'][k]['cv']['PR_AUC'][0]:.4f} ± {RESULTS['E3'][k]['cv']['PR_AUC'][1]:.4f} | CV MCC {RESULTS['E3'][k]['cv']['MCC'][0]:.4f} ± {RESULTS['E3'][k]['cv']['MCC'][1]:.4f}")
json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)

ET          CV PR-AUC 0.3868 ± 0.0196 | CV MCC 0.3367 ± 0.0286
RF          CV PR-AUC 0.3903 ± 0.0181 | CV MCC 0.3446 ± 0.0204
LR          CV PR-AUC 0.3061 ± 0.0176 | CV MCC 0.2950 ± 0.0211
ET_summary  CV PR-AUC 0.3515 ± 0.0241 | CV MCC 0.3237 ± 0.0219


### 6.2 Feature importance (ET)

In [15]:
# importance grouped into summary / 2014-15 / 2016
imp = pd.Series(et_final.feature_importances_, index=feat_names)
n_s = summ.shape[1]
groups = {"Summary features (11)": imp.iloc[-n_s:].sum(), "Daily readings 2014-2015 (730)": imp.iloc[:730].sum(), "Daily readings 2016 (305)": imp.iloc[730:-n_s].sum()}
print(pd.Series(groups).round(3)); print(imp.sort_values(ascending=False).head(10).round(4))
RESULTS["E3"]["ET"]["importance_groups"] = {k: float(v) for k, v in groups.items()}
RESULTS["E3"]["ET"]["top_features"] = {k: float(v) for k, v in imp.sort_values(ascending=False).head(10).items()}

Summary features (11)             0.023
Daily readings 2014-2015 (730)    0.490
Daily readings 2016 (305)         0.487
dtype: float64
2016-09-17    0.0097
2016-09-15    0.0092
2016-09-18    0.0076
2016-09-19    0.0075
2016-09-16    0.0070
mad_log       0.0063
2016-09-20    0.0059
2016-09-21    0.0052
2016-10-06    0.0047
2016-10-31    0.0047
dtype: float64


## 7. Comparison with the paper + figures

In [16]:
paper = dict(MCC=0.9506, ROC_AUC=0.9965, Recall=0.98, Precision=0.97, FPR=0.98 * (1 - 0.97) / 0.97)   # FPR implied by recall and precision on a 50/50 test
RESULTS["paper"] = paper
# comparison table
rows = [("Paper (reported)", paper), ("E1 authors' pipeline, 50/50 test", m_e1), ("E1 model, real customers only", m_e1_real),
        ("E2 split first, real-ratio test", m_e2), ("E3 Extra Trees (corrected)", m_et)] + \
       [(f"E3 {dict(RF='Random Forest', LR='Logistic Regression', ET_summary='ET, summary features only')[k]}", v["test"]) for k, v in others.items()]
tab = pd.DataFrame([{**{"Experiment": n}, **{k: m.get(k, np.nan) for k in ["MCC", "PR_AUC", "ROC_AUC", "Precision", "Recall", "FPR", "F1", "Accuracy", "Prec_top5"]}} for n, m in rows])
print(tab.round(4).to_string(index=False))
tab.to_csv(f"{OUT}/comparison.csv", index=False)
json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)

                      Experiment    MCC  PR_AUC  ROC_AUC  Precision  Recall    FPR     F1  Accuracy  Prec_top5
                Paper (reported) 0.9506     NaN   0.9965     0.9700  0.9800 0.0303    NaN       NaN        NaN
E1 authors' pipeline, 50/50 test 0.9312  0.9931   0.9915     0.9686  0.9624 0.0312 0.9655    0.9656     1.0000
   E1 model, real customers only 0.8344  0.9667   0.9925     0.7392  0.9759 0.0312 0.8412    0.9694     0.9925
 E2 split first, real-ratio test 0.2951  0.3411   0.7793     0.4940  0.2291 0.0229 0.3130    0.9106     0.4417
      E3 Extra Trees (corrected) 0.3479  0.3775   0.7996     0.4163  0.3925 0.0537 0.4040    0.8970     0.5161
                E3 Random Forest 0.3424  0.3911   0.8002     0.4072  0.3925 0.0558 0.3997    0.8952     0.5261
          E3 Logistic Regression 0.2709  0.2785   0.7540     0.3194  0.3617 0.0752 0.3392    0.8747     0.3945
    E3 ET, summary features only 0.3173  0.3401   0.7815     0.3870  0.3659 0.0566 0.3762    0.8921     0.4417


In [17]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 200})
# fig 1: MCC bars and PR curves
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
names = ["Paper", "E1\n(leaky)", "E2\n(split first)", "E3 ET", "E3 RF", "E3 LR"]
mccs = [paper["MCC"], m_e1["MCC"], m_e2["MCC"], m_et["MCC"], others["RF"]["test"]["MCC"], others["LR"]["test"]["MCC"]]
cols = ["#999999", "#D9534F", "#2A78D6", "#2A78D6", "#2A78D6", "#2A78D6"]
ax[0].bar(names, mccs, color=cols)
for i, v in enumerate(mccs): ax[0].text(i, v + 0.015, f"{v:.2f}", ha="center", fontsize=8)
ax[0].set_ylim(0, 1.08); ax[0].set_title("MCC on each test set", loc="left")
for lo_, hi_ in [(0.75, 0.84)]: ax[0].axhspan(lo_, hi_, color="#EEEEEE", zorder=0)
ax[0].text(5.4, 0.795, "earlier\nstudies", fontsize=7, va="center", ha="right", color="#555")
for name_, p_, c_ in [("E2 split first", p2, "#7FA7D9"), ("E3 Extra Trees", p_et, "#2A78D6"), ("E3 Random Forest", np.load(f"{OUT}/p_E3_RF.npy"), "#54A24B"), ("E3 Logistic Reg.", np.load(f"{OUT}/p_E3_LR.npy"), "#B279A2")]:
    pr, rc, _ = precision_recall_curve(yte, p_)
    ax[1].plot(rc, pr, color=c_, lw=1.5, label=f"{name_} (AP {average_precision_score(yte, p_):.2f})")
ax[1].axhline(yte.mean(), color="#999", ls="--", lw=0.8); ax[1].text(0.01, yte.mean() + 0.02, "random model (0.089)", ha="left", fontsize=7, color="#555")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-recall curves, real-ratio test set", loc="left"); ax[1].legend(frameon=False, fontsize=7.5)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_compare.png"); plt.close()

# fig 2: confusion matrices
fig, ax = plt.subplots(1, 3, figsize=(10, 3.1))
for a_, (t_, m_) in zip(ax, [("E1 (leaky, 50/50 test)", m_e1), ("E2 (split first)", m_e2), ("E3 Extra Trees", m_et)]):
    cm = np.array([[m_["TN"], m_["FP"]], [m_["FN"], m_["TP"]]])
    a_.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2): a_.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=10, color="white" if cm[i, j] / cm[i].sum() > 0.5 else "black")
    a_.set_xticks([0, 1]); a_.set_xticklabels(["Normal", "Theft"]); a_.set_yticks([0, 1]); a_.set_yticklabels(["Normal", "Theft"])
    a_.set_xlabel("Predicted"); a_.set_ylabel("Actual"); a_.set_title(t_, loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_confusion.png"); plt.close()

# fig 3: CV vs test PR-AUC
fig, ax = plt.subplots(figsize=(7, 3.2))
lab = ["Extra Trees", "Random Forest", "Logistic Reg.", "ET summary only"]
keys = ["ET", "RF", "LR", "ET_summary"]
mu_ = [RESULTS["E3"][k]["cv"]["PR_AUC"][0] for k in keys]; sd_ = [RESULTS["E3"][k]["cv"]["PR_AUC"][1] for k in keys]
te_ = [RESULTS["E3"][k]["test"]["PR_AUC"] for k in keys]
x_ = np.arange(4)
ax.bar(x_ - 0.18, mu_, 0.36, yerr=sd_, color="#2A78D6", capsize=3, label="5-fold CV (mean ± SD)")
ax.bar(x_ + 0.18, te_, 0.36, color="#A9C9F0", label="Held-out test")
ax.axhline(yte.mean(), color="#999", ls="--", lw=0.8); ax.text(3.45, yte.mean() + 0.008, "random model", ha="right", fontsize=7, color="#555"); ax.set_xticks(x_); ax.set_xticklabels(lab); ax.set_ylabel("PR-AUC")
ax.set_ylim(0, 0.48); ax.set_title("E3 models: cross-validation agrees with the test set", loc="left"); ax.legend(frameon=False, fontsize=8, loc="upper right", ncol=2)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_cv.png"); plt.close()
log("done")

[   2830s] done
